# **Data and Information Quality Project**

In [137]:
import pandas as pd
from ydata_profiling import ProfileReport
import numpy as np
import json
import os
import re

In [138]:
DATASET = pd.read_csv('Comune-di-Milano-Servizi-alla-persona-parrucchieri-estetisti(in).csv',sep=';',encoding='unicode_escape')


In [139]:
DATASET

,Tipo esercizio pa,Ubicazione,Tipo via,Via,Civico,Codice via,ZD,Prevalente,Superficie altri usi,Superficie lavorativa
0,NaN,LGO DEI GELSOMINI N. 10 (z.d. 6),LGO,DEI GELSOMINI,10,5394.0,6,NaN,NaN,55.0
1,NaN,PZA FIDIA N. 3 (z.d. 9),PZA,FIDIA,3,1144.0,9,CENTRO MASSAGGI RILASSANTI NON ESTETICI,2.0,28.0
2,NaN,VIA ADIGE N. 10 (z.d. 5),VIA,ADIGE,10,4216.0,5,CENTRO BENESSERE,2.0,27.0
3,NaN,VIA BARACCHINI FLAVIO N. 9 (z.d. 1),VIA,BARACCHINI FLAVIO,9,356.0,1,TRUCCO SEMIPERMANENTE,NaN,NaN
4,NaN,VIA BERGAMO N. 12 (z.d. 4),VIA,BERGAMO,12,3189.0,4,NaN,NaN,50.0
...,...,...,...,...,...,...,...,...,...,...
3904,TIPO D ESTET.APPAR.ELETTROMECC;TIPO C TRATT.ES...,VIA SARPI FRA' PAOLO N. 1 con ingr.da v.le mon...,VIA,SARPI FRA' PAOLO,1,7210.0,1,NaN,NaN,NaN
3905,TIPO D ESTET.APPAR.ELETTROMECC;TIPO C TRATT.ES...,CSO DI PORTA TICINESE N. 4 ; (z.d. 1),CSO,DI PORTA TICINESE,4,541.0,1,NaN,NaN,65.0
3906,TIPO D ESTET.APPAR.ELETTROMECC;TIPO C TRATT.ES...,VIA CANDOGLIA N. 2 ; (z.d. 9),VIA,CANDOGLIA,2,1518.0,9,NaN,NaN,NaN
3907,TIPO D ESTET.APPAR.ELETTROMECC;TIPO C TRATT.ES...,VIA NIRONE num.002a; (z.d. 1),VIA,NIRONE,NaN,640.0,1,NaN,NaN,NaN


# *Data Quality Assesement*


In [140]:
#Data information
DATASET.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3909 entries, 0 to 3908
Data columns (total 10 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Tipo esercizio pa      3878 non-null   object 
 1   Ubicazione             3909 non-null   object 
 2   Tipo via               3908 non-null   object 
 3   Via                    3908 non-null   object 
 4   Civico                 3832 non-null   object 
 5   Codice via             3908 non-null   float64
 6   ZD                     3908 non-null   object 
 7   Prevalente             294 non-null    object 
 8   Superficie altri usi   745 non-null    float64
 9   Superficie lavorativa  2601 non-null   float64
dtypes: float64(3), object(7)
memory usage: 305.5+ KB


##### Single column analisys

In [141]:
# Distinct values for each column
DATASET.nunique()

Tipo esercizio pa         103
Ubicazione               3554
Tipo via                   17
Via                      1370
Civico                    235
Codice via               1376
ZD                         10
Prevalente                 63
Superficie altri usi       52
Superficie lavorativa     145
dtype: int64

In [142]:
# Uniqueness percentage for each column
UNIQUENESS = (DATASET.nunique() / DATASET.shape[0]) * 100
UNIQUENESS

Tipo esercizio pa         2.634945
Ubicazione               90.918393
Tipo via                  0.434894
Via                      35.047327
Civico                    6.011768
Codice via               35.200819
ZD                        0.255820
Prevalente                1.611665
Superficie altri usi      1.330263
Superficie lavorativa     3.709389
dtype: float64

In [143]:
#Information about the type of esercizio
DATASET.value_counts("Tipo esercizio pa")

Tipo esercizio pa
Parrucchiere per signora                                         1048
ACCONCIATORE                                                      586
Parrucchiere per uomo                                             439
TIPO A - REG.2003                                                 335
TIPO A - REG.2003;TIPO B CENTRO DI ABBRONZATURA                   313
                                                                 ... 
TIPO A-B-C-D;ACCONCIATORE                                           1
TIPO A-B-C-D;Acconciatore                                           1
TIPO A-B-C-D;Estetista in profumeria                                1
TIPO D ESTET.APPAR.ELETTROMECC;TIPO C TRATT.ESTETICI DIMAGRIM       1
Truccatore                                                          1
Name: count, Length: 103, dtype: int64

##### Completeness

In [144]:
# For each column
NULL_VALUES = DATASET.isnull().sum()
NOT_NULL_VALUES = DATASET.notnull().sum()
ROWS = DATASET.shape[0]
COMPLETENESS = NOT_NULL_VALUES / ROWS
COMPLETENESS = COMPLETENESS.map('{:.2%}'.format)
COMPLETENESS

Tipo esercizio pa         99.21%
Ubicazione               100.00%
Tipo via                  99.97%
Via                       99.97%
Civico                    98.03%
Codice via                99.97%
ZD                        99.97%
Prevalente                 7.52%
Superficie altri usi      19.06%
Superficie lavorativa     66.54%
dtype: object

In [145]:
# For the entire dataset
TOT_NULL_VALUES = DATASET.isnull().sum().sum()   # TODO: Controllare se abbiamo celle null con valori differenti
TOT_NOT_NULL_VALUES = DATASET.notnull().sum().sum()
TOT_COMPLETENESS = TOT_NOT_NULL_VALUES / DATASET.size
TOT_COMPLETENESS = '{:.2%}'.format(TOT_COMPLETENESS)
TOT_COMPLETENESS

'79.03%'

##### Duplication

In [146]:
DATASET.duplicated().any()

np.True_

In [147]:
DATASET[DATASET.duplicated()]

,Tipo esercizio pa,Ubicazione,Tipo via,Via,Civico,Codice via,ZD,Prevalente,Superficie altri usi,Superficie lavorativa
88,Acconciatore,VIA CORREGGIO N. 8 (z.d. 7),VIA,CORREGGIO,8,6287.0,7,ACCONCIATORE,NaN,NaN


# *Data Profiling*


In [148]:
#profile = ProfileReport(DATASET, title=" Report comune di Milano Servizi alla persona di parrucchieri e estetisti")
#profile.to_file("Report comune di Milano Servizi alla persona di parrucchieri e estetisti.html")

# *Data Wrangling*

Renaming and Sorting

In [149]:
DATASET.rename(columns={"Tipo esercizio pa":"Tipo esercizio", "Prevalente":"Attivita Primaria", "ZD":"Municipio"},inplace=True)

In [150]:
DATASET = DATASET.sort_values(by = ['Tipo esercizio', "Ubicazione"], ascending=True)
DATASET.head()

,Tipo esercizio,Ubicazione,Tipo via,Via,Civico,Codice via,Municipio,Attivita Primaria,Superficie altri usi,Superficie lavorativa
33,(z.d. 9),CSO,COMO,15,1111,9.0,ACCONCIATORE,NaN,195.0,NaN
208,ACCONCIATORE,ALZ NAVIGLIO PAVESE N. 52 ; (z.d. 6),ALZ,NAVIGLIO PAVESE,52,5161.0,6,NaN,2.0,12.0
209,ACCONCIATORE,BST DI PORTA VOLTA N. 13 ; (z.d. 1),BST,DI PORTA VOLTA,13,1066.0,1,NaN,NaN,NaN
211,ACCONCIATORE,CSO BUENOS AIRES N. 64 ; (z.d. 3),CSO,BUENOS AIRES,64,2129.0,3,NaN,NaN,57.0
212,ACCONCIATORE,CSO COLOMBO CRISTOFORO N. 8 ; (z.d. 6),CSO,COLOMBO CRISTOFORO,8,5114.0,6,NaN,NaN,41.0


Standardization

In [151]:
def to_upper_safe(x):
    if isinstance(x, str):
        return x.upper()
    return x

DATASET = DATASET.map(lambda x: to_upper_safe(x) if pd.notnull(x) else x)

In [152]:
# Transform "Tipo Esercizio"

new_cols = DATASET["Tipo esercizio"].str.split(";", expand=True)
new_cols = new_cols.apply(lambda col: col.str.strip())
new_cols = new_cols.replace("", np.nan)
new_cols = new_cols.replace({None: np.nan})
new_cols.columns = [f"Tipo_esercizio_{i+1}" for i in range(new_cols.shape[1])]

DATASET = pd.concat([DATASET, new_cols], axis=1)

split_cols = DATASET[[c for c in DATASET.columns if c.startswith("Tipo_esercizio_")]]

acconciatori = {"ACCONCIATORE", "PARRUCCHIERE MISTO", "PARRUCCHIERE PER SIGNORA", "PARRUCCHIERE PER UOMO", "BARBIERE"}
estetisti = {"ESTETISTA", "ESTETISTA IN PROFUMERIA", "TIPO A - REG.2003", "TIPO A ESTETICA MANUALE", "TIPO A-B-C-D", "MANICURE", "PEDICURE ESTETICO", "TRUCCATORE"}
centri_abbronzatura = {"CENTRO ABBRONZATURA", "TIPO B CENTRO DI ABBRONZATURA", "CENTRO BENESSERE", "CENTRO MASSAGGI", "TIPO A-B-C-D"}
trattamenti = {"TIPO C TRATT.ESTETICI DIMAGRIM", "TIPO D ESTET.APPAR.ELETTROMECC", "TIPO A-B-C-D"}
tatuaggi = {"ESECUZIONE DI TATUAGGI E PIERCING"}

def has_any_from_group(group_set):
    return split_cols.isin(group_set).any(axis=1)

DATASET["ACCONCIATORE"] = has_any_from_group(acconciatori)
DATASET["ESTETISTA"] = has_any_from_group(estetisti)
DATASET["CENTRO ABBRONZATURA"] = has_any_from_group(centri_abbronzatura)
DATASET["TRATTAMENTO"] = has_any_from_group(trattamenti)
DATASET["TATUAGGI E PIERCING"] = has_any_from_group(tatuaggi)

DATASET = DATASET.drop(columns=split_cols.columns)
DATASET

,Tipo esercizio,Ubicazione,Tipo via,Via,Civico,Codice via,Municipio,Attivita Primaria,Superficie altri usi,Superficie lavorativa,ACCONCIATORE,ESTETISTA,CENTRO ABBRONZATURA,TRATTAMENTO,TATUAGGI E PIERCING
33,(Z.D. 9),CSO,COMO,15,1111,9.0,ACCONCIATORE,NaN,195.0,NaN,False,False,False,False,False
208,ACCONCIATORE,ALZ NAVIGLIO PAVESE N. 52 ; (Z.D. 6),ALZ,NAVIGLIO PAVESE,52,5161.0,6,NaN,2.0,12.0,True,False,False,False,False
209,ACCONCIATORE,BST DI PORTA VOLTA N. 13 ; (Z.D. 1),BST,DI PORTA VOLTA,13,1066.0,1,NaN,NaN,NaN,True,False,False,False,False
211,ACCONCIATORE,CSO BUENOS AIRES N. 64 ; (Z.D. 3),CSO,BUENOS AIRES,64,2129.0,3,NaN,NaN,57.0,True,False,False,False,False
212,ACCONCIATORE,CSO COLOMBO CRISTOFORO N. 8 ; (Z.D. 6),CSO,COLOMBO CRISTOFORO,8,5114.0,6,NaN,NaN,41.0,True,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26,NaN,VLE CORSICA N. 91 (Z.D. 4),VLE,CORSICA,91,3262.0,4,NaN,NaN,45.0,False,False,False,False,False
27,NaN,VLE SABOTINO N. 5 (Z.D. 5),VLE,SABOTINO,5,4046.0,5,NaN,NaN,51.0,False,False,False,False,False
28,NaN,VLE SABOTINO N. 8 (Z.D. 5),VLE,SABOTINO,8,4046.0,5,ESTETICA A,NaN,46.0,False,False,False,False,False
29,NaN,VLE SUZZANI GIOVANNI N. 19 (Z.D. 9),VLE,SUZZANI GIOVANNI,19,1446.0,9,NaN,NaN,66.0,False,False,False,False,False


Column Splitting

In [154]:
# Split "Ubicazione"
df = DATASET.copy()

# Normalization for Ubicazione
df['Ubicazione'] = df['Ubicazione'].str.replace('NUM', 'N')

# Use a different name for the DATASET for not change the original one

# Splitting the "Ubicazione" column on "N." to separate address and addition data
split_column = df['Ubicazione'].str.split(r"\bN\.", n=1, expand=True)
split_column.columns = ['Ubicazione_effettiva', 'Ubicazione_data']
# Clean the Ubicazione_data by stripping leading/trailing whitespace
split_column['Ubicazione_data'] = split_column['Ubicazione_data'].str.strip()
# Add split columns to df
df = pd.concat([df, split_column], axis=1)

# Splitting the "Ubicazione_data" column on "(" to separate civico and z.d.
split_column_2 = df['Ubicazione_data'].str.split(r"\(Z\.D\.", n=1, expand=True)
split_column_2.columns = ['Civico_to_check', 'Municipio_to_check']
# Add to original database
df = pd.concat([df, split_column_2], axis=1)
# Remove ";" from "Civico_to_check"
df['Civico_to_check'] = df['Civico_to_check'].str.replace(";", "", regex=False)
df['Civico_to_check'] = df['Civico_to_check'].str.strip()
# Remove the closing parenthesis ")" and the ZD description from the "Municipio_to_check" column
df["Municipio_to_check"] = df["Municipio_to_check"].str.replace("Z.D.", "", regex=False)
df["Municipio_to_check"] = df["Municipio_to_check"].str.replace(")", "", regex=False)
df['Municipio_to_check'] = df['Municipio_to_check'].str.strip()
# Convert null/empty values to pd.NA
df['Civico_to_check'] = df['Civico_to_check'].fillna(pd.NA)
df['Municipio_to_check'] = df['Municipio_to_check'].fillna(pd.NA)

# Splitting the "Ubicazione_effettiva" column on the first " " to separate "Tipo via" and "Via"
split_column_3 = df['Ubicazione_effettiva'].str.split(" ", n=1, expand=True)
split_column_3.columns = ['Tipo via_to_check', 'Via_to_check']
# Add to original database
df = pd.concat([df, split_column_3], axis=1)
# Convert null/empty values to pd.NA for Tipo via and Via
df['Tipo via_to_check'] = df['Tipo via_to_check'].fillna(pd.NA)
df['Tipo via_to_check'] = df['Tipo via_to_check'].str.strip()
df['Via_to_check'] = df['Via_to_check'].fillna(pd.NA)
df['Via_to_check'] = df['Via_to_check'].str.strip()

df = df.drop(['Ubicazione_effettiva', 'Ubicazione_data'], axis=1)
df[["Ubicazione", "Tipo via_to_check", "Via_to_check", "Civico_to_check", "Municipio_to_check"]]

,Ubicazione,Tipo via_to_check,Via_to_check,Civico_to_check,Municipio_to_check
33,CSO,CSO,<NA>,<NA>,<NA>
208,ALZ NAVIGLIO PAVESE N. 52 ; (Z.D. 6),ALZ,NAVIGLIO PAVESE,52,6
209,BST DI PORTA VOLTA N. 13 ; (Z.D. 1),BST,DI PORTA VOLTA,13,1
211,CSO BUENOS AIRES N. 64 ; (Z.D. 3),CSO,BUENOS AIRES,64,3
212,CSO COLOMBO CRISTOFORO N. 8 ; (Z.D. 6),CSO,COLOMBO CRISTOFORO,8,6
...,...,...,...,...,...
26,VLE CORSICA N. 91 (Z.D. 4),VLE,CORSICA,91,4
27,VLE SABOTINO N. 5 (Z.D. 5),VLE,SABOTINO,5,5
28,VLE SABOTINO N. 8 (Z.D. 5),VLE,SABOTINO,8,5
29,VLE SUZZANI GIOVANNI N. 19 (Z.D. 9),VLE,SUZZANI GIOVANNI,19,9


Civico and Municipio

In [155]:
# Check if the civico is equal to the civico_to_check, return only the rows where they are not equal
notok = df[(df['Civico'] != df['Civico_to_check'])]
notok

,Tipo esercizio,Ubicazione,Tipo via,Via,Civico,Codice via,Municipio,Attivita Primaria,Superficie altri usi,Superficie lavorativa,...,TRATTAMENTO,TATUAGGI E PIERCING,tipo_via_check,indirizzo_check,civico_check,municipio_check,Civico_to_check,Municipio_to_check,Tipo via_to_check,Via_to_check
33,(Z.D. 9),CSO,COMO,15,1111,9.0,ACCONCIATORE,NaN,195.0,NaN,...,False,False,NaN,NaN,NaN,NaN,<NA>,<NA>,CSO,<NA>
215,ACCONCIATORE,CSO DI PORTA ROMANA N. 117 PIANO T. E INTERRAT...,CSO,DI PORTA ROMANA,117,402.0,1,NaN,NaN,NaN,...,False,False,CSO,DI PORTA ROMANA,117,1,117 PIANO T. E INTERRATO,1,CSO,DI PORTA ROMANA
224,ACCONCIATORE,CSO LODI N. 112 1Ø PIANO; (Z.D. 4),CSO,LODI,112,4068.0,4,NaN,NaN,NaN,...,False,False,CSO,LODI,112,4,112 1Ø PIANO,4,CSO,LODI
232,ACCONCIATORE,CSO VITTORIO EMANUELE II N. 15 1Ø PIANO; (Z.D. 1),CSO,VITTORIO EMANUELE II,15,214.0,1,NaN,NaN,NaN,...,False,False,CSO,VITTORIO EMANUELE II,15,1,15 1Ø PIANO,1,CSO,VITTORIO EMANUELE II
233,ACCONCIATORE,CSO VITTORIO EMANUELE II N. 24 2ØINTERRATO; (Z...,CSO,VITTORIO EMANUELE II,24,214.0,1,NaN,NaN,42.0,...,False,False,CSO,VITTORIO EMANUELE II,24,1,24 2ØINTERRATO,1,CSO,VITTORIO EMANUELE II
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3896,TIPO C TRATT.ESTETICI DIMAGRIM,VIA DURER ALBERTO N. 4 PRIMO PIANO; (Z.D. 7),VIA,DURER ALBERTO,4,6602.0,7,NaN,NaN,43.0,...,True,False,VIA,DURER ALBERTO,4,7,4 PRIMO PIANO,7,VIA,DURER ALBERTO
3897,TIPO C TRATT.ESTETICI DIMAGRIM,VIA ZURETTI GIANFRANCO N. 31 AL SECONDO PIANO ...,VIA,ZURETTI GIANFRANCO,31,1216.0,2,NaN,NaN,NaN,...,True,False,VIA,ZURETTI GIANFRANCO,31,2,31 AL SECONDO PIANO SCALA A,2,VIA,ZURETTI GIANFRANCO
3904,TIPO D ESTET.APPAR.ELETTROMECC;TIPO C TRATT.ES...,VIA SARPI FRA' PAOLO N. 1 CON INGR.DA V.LE MON...,VIA,SARPI FRA' PAOLO,1,7210.0,1,NaN,NaN,NaN,...,True,False,VIA,SARPI FRA' PAOLO,1,1,1 CON INGR.DA V.LE MONTELLO 4/6,1,VIA,SARPI FRA' PAOLO
3907,TIPO D ESTET.APPAR.ELETTROMECC;TIPO C TRATT.ES...,VIA NIRONE N.002A; (Z.D. 1),VIA,NIRONE,NaN,640.0,1,NaN,NaN,NaN,...,True,False,NaN,NaN,NaN,NaN,002A,1,VIA,NIRONE


In [156]:
# Clean and convert Civico_to_check and ZD_to_check before comparison
def clean_numeric_string(value):
    if pd.isna(value):
        return pd.NA, pd.NA
    
    if isinstance(value, str):
        value_str = value.strip()
        if not value_str:
            return pd.NA, pd.NA
        
        # Check if value contains at least one digit, otherwise return pd.NA
        if not re.search(r'\d', value_str):
            return pd.NA, pd.NA
        
        # Find all numeric sequences with optional suffixes (like 7, 0061, 22c, 6/2)
        all_numbers = re.findall(r'\d+(?:[a-zA-Z](?![a-zA-Z])|/\d*)?', value_str)
        
        if not all_numbers:
            return pd.NA, value
        
        # Look for numbers with leading zeros (e.g., 0061, 008, 002a)
        numbers_with_leading_zeros = [num for num in all_numbers if re.match(r'^0\d+', num)]
        
        # Prioritize numbers with leading zeros, otherwise take the first number
        if numbers_with_leading_zeros:
            numeric_part = numbers_with_leading_zeros[0]
        else:
            numeric_part = all_numbers[0]
        
        # Find the position of the selected number and extract remaining text after it
        num_position = value_str.find(numeric_part)
        remaining = value_str[num_position + len(numeric_part):].strip()
        additional_part = remaining if remaining else pd.NA
        
        # Remove leading zeros but keep at least one digit, preserving suffixes like /2, a, b, etc.
        numeric_part = re.sub(r'^0+(?=\d)', '', numeric_part)
        if not numeric_part:
            numeric_part = "0"
        
        return numeric_part, additional_part
        
    elif isinstance(value, (int, float)):
        return str(int(value)), pd.NA
    
    return pd.NA, pd.NA

# Apply cleaning to Civico_to_check and save both parts
df[['Civico_to_check', 'Civico_additional']] = df['Civico_to_check'].apply(
    lambda x: pd.Series(clean_numeric_string(x))
)
df[['Municipio_to_check', 'Municipio_additional']] = df['Municipio_to_check'].apply(
    lambda x: pd.Series(clean_numeric_string(x))
)

# Convert to string for comparison, replacing pd.NA with empty string
df['Civico'] = df['Civico'].astype(str).replace('<NA>', '0').replace('nan', '0')
df['Municipio'] = df['Municipio'].astype(str).replace('<NA>', '0').replace('nan', '0')
df['Civico_to_check'] = df['Civico_to_check'].astype(str).replace('<NA>', '0')
df['Municipio_to_check'] = df['Municipio_to_check'].astype(str).replace('<NA>', '0')
# Check the not right values
condition = ((df['Civico_to_check'] != df['Civico']) | (df['Municipio_to_check'] != df['Municipio']))
check_civico_municipio = df[condition][['Civico', 'Civico_to_check', 'Civico_additional', 'Municipio', 'Municipio_to_check', 'Municipio_additional']]
check_civico_municipio

,Civico,Civico_to_check,Civico_additional,Municipio,Municipio_to_check,Municipio_additional
33,1111,0,<NA>,ACCONCIATORE,0,<NA>
280,7,7,<NA>,7,5,<NA>
286,0,77H,<NA>,8,8,<NA>
308,0,161A,<NA>,8,8,<NA>
317,0,6C,<NA>,5,5,<NA>
...,...,...,...,...,...,...
3114,22,562,<NA>,3,3,<NA>
3148,0,46,<NA>,1,1,<NA>
2946,0,1214,<NA>,1,1,<NA>
3839,0,11D,<NA>,6,6,<NA>


# *Error Detection & Correction*

In [157]:
# Fix "Ubicazione"

# Fix Civico_to_check and Municipio_to_check
df.loc[:, 'Civico'] = df.apply(
    lambda row: row["Civico_to_check"] if row["Civico"] == 0 and row["Civico_to_check"] != 0 else 
                (row["Civico"] if row["Civico_to_check"] == 0 else 
                (row["Civico_to_check"] if row["Civico"] != row["Civico_to_check"] else row["Civico"])), axis=1)
df.loc[:, 'Municipio'] = df.apply(
    lambda row: row["Municipio_to_check"] if row["Municipio"] == 0 and row["Municipio_to_check"] != 0 else 
                (row["Municipio"] if row["Municipio_to_check"] == 0 else 
                (row["Municipio_to_check"] if row["Municipio"] != row["Municipio_to_check"] else row["Municipio"])), axis=1)

df.iloc[check_civico_municipio.index][["Civico", "Civico_to_check", "Civico_additional", "Municipio", "Municipio_to_check", "Municipio_additional"]]

,Civico,Civico_to_check,Civico_additional,Municipio,Municipio_to_check,Municipio_additional
241,1,1,<NA>,1,1,<NA>
489,30,30,<NA>,8,8,<NA>
495,48,48,<NA>,4,4,<NA>
517,6,6,<NA>,8,8,<NA>
527,41,41,<NA>,1,1,<NA>
...,...,...,...,...,...,...
3434,14,14,VIA BRENTANO,1,1,<NA>
3453,25,25,<NA>,4,4,<NA>
3251,52,52,<NA>,5,5,<NA>
3886,66,66,<NA>,1,1,<NA>


In [158]:
# Maybe other fixes

# *Null Values Handling*

In [159]:
# Drop rows where "Attivita Primaria" and "Tipo esercizio" are null
DATASET = DATASET.dropna(subset=["Attivita Primaria", "Tipo esercizio"], how='all')

In [160]:
# If "Attivita Primaria" is null, fill with name of the first True column of "Tipo esercizio"

# Maybe review this part --> maybe we are generalizing too much and losing information

tipo_bool_cols = ["ACCONCIATORE", "ESTETISTA", "CENTRO ABBRONZATURA", "TRATTAMENTO", "TATUAGGI E PIERCING"]
tipo_bool = DATASET[tipo_bool_cols]

has_true = tipo_bool.any(axis=1)

first_true_col = tipo_bool.idxmax(axis=1)

first_true_col = first_true_col.where(has_true, np.nan)

mask_att_null = DATASET["Attivita Primaria"].isna()

DATASET.loc[mask_att_null, "Attivita Primaria"] = first_true_col[mask_att_null]

DATASET = DATASET.drop(columns='Tipo esercizio')

In [161]:
DATASET

,Ubicazione,Tipo via,Via,Civico,Codice via,Municipio,Attivita Primaria,Superficie altri usi,Superficie lavorativa,ACCONCIATORE,ESTETISTA,CENTRO ABBRONZATURA,TRATTAMENTO,TATUAGGI E PIERCING,tipo_via_check,indirizzo_check,civico_check,municipio_check
33,CSO,COMO,15,1111,9.0,ACCONCIATORE,NaN,195.0,NaN,False,False,False,False,False,NaN,NaN,NaN,NaN
208,ALZ NAVIGLIO PAVESE N. 52 ; (Z.D. 6),ALZ,NAVIGLIO PAVESE,52,5161.0,6,ACCONCIATORE,2.0,12.0,True,False,False,False,False,ALZ,NAVIGLIO PAVESE,52,6
209,BST DI PORTA VOLTA N. 13 ; (Z.D. 1),BST,DI PORTA VOLTA,13,1066.0,1,ACCONCIATORE,NaN,NaN,True,False,False,False,False,BST,DI PORTA VOLTA,13,1
211,CSO BUENOS AIRES N. 64 ; (Z.D. 3),CSO,BUENOS AIRES,64,2129.0,3,ACCONCIATORE,NaN,57.0,True,False,False,False,False,CSO,BUENOS AIRES,64,3
212,CSO COLOMBO CRISTOFORO N. 8 ; (Z.D. 6),CSO,COLOMBO CRISTOFORO,8,5114.0,6,ACCONCIATORE,NaN,41.0,True,False,False,False,False,CSO,COLOMBO CRISTOFORO,8,6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21,VIA TENCA CARLO N. 10 (Z.D. 2),VIA,TENCA CARLO,10,2118.0,2,TATUAGGI E PIERCING,52.0,47.0,False,False,False,False,False,VIA,TENCA CARLO,10,2
24,VIA WATT GIACOMO N. 5 (Z.D. 6),VIA,WATT GIACOMO,5,5292.0,6,CENTRO MASSAGGI RILASSANTI NON ESTETICI,NaN,NaN,False,False,False,False,False,VIA,WATT GIACOMO,5,6
25,VLE BLIGNY N. 23 (Z.D. 5),VLE,BLIGNY,23,4045.0,5,SERVIZI DI CENTRI PER IL BENESSERE FISICO,NaN,149.0,False,False,False,False,False,VLE,BLIGNY,23,5
28,VLE SABOTINO N. 8 (Z.D. 5),VLE,SABOTINO,8,4046.0,5,ESTETICA A,NaN,46.0,False,False,False,False,False,VLE,SABOTINO,8,5


In [162]:
# Fill "Superficie lavorativa" with median value grouping by "Tipo esercizio"
group_median = (
    DATASET
    .groupby(tipo_bool_cols)["Superficie lavorativa"]
    .transform("median")   # median is computed ignoring NaN
)

mask_superficie_null = DATASET["Superficie lavorativa"].isna()
DATASET.loc[mask_superficie_null, "Superficie lavorativa"] = group_median[mask_superficie_null]

In [163]:
# Fill "Superficie altri usi" with 0
DATASET["Superficie altri usi"] = DATASET["Superficie altri usi"].fillna(0)

In [164]:
# Fill "Codice via" with values extracted from "Ubicazione" if existing, 0 otherwise

In [165]:
# Fill "Municipio" with values extracted from "Ubicazione" if existing, 0 otherwise

In [166]:
DATASET

,Ubicazione,Tipo via,Via,Civico,Codice via,Municipio,Attivita Primaria,Superficie altri usi,Superficie lavorativa,ACCONCIATORE,ESTETISTA,CENTRO ABBRONZATURA,TRATTAMENTO,TATUAGGI E PIERCING,tipo_via_check,indirizzo_check,civico_check,municipio_check
33,CSO,COMO,15,1111,9.0,ACCONCIATORE,NaN,195.0,34.0,False,False,False,False,False,NaN,NaN,NaN,NaN
208,ALZ NAVIGLIO PAVESE N. 52 ; (Z.D. 6),ALZ,NAVIGLIO PAVESE,52,5161.0,6,ACCONCIATORE,2.0,12.0,True,False,False,False,False,ALZ,NAVIGLIO PAVESE,52,6
209,BST DI PORTA VOLTA N. 13 ; (Z.D. 1),BST,DI PORTA VOLTA,13,1066.0,1,ACCONCIATORE,0.0,30.0,True,False,False,False,False,BST,DI PORTA VOLTA,13,1
211,CSO BUENOS AIRES N. 64 ; (Z.D. 3),CSO,BUENOS AIRES,64,2129.0,3,ACCONCIATORE,0.0,57.0,True,False,False,False,False,CSO,BUENOS AIRES,64,3
212,CSO COLOMBO CRISTOFORO N. 8 ; (Z.D. 6),CSO,COLOMBO CRISTOFORO,8,5114.0,6,ACCONCIATORE,0.0,41.0,True,False,False,False,False,CSO,COLOMBO CRISTOFORO,8,6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21,VIA TENCA CARLO N. 10 (Z.D. 2),VIA,TENCA CARLO,10,2118.0,2,TATUAGGI E PIERCING,52.0,47.0,False,False,False,False,False,VIA,TENCA CARLO,10,2
24,VIA WATT GIACOMO N. 5 (Z.D. 6),VIA,WATT GIACOMO,5,5292.0,6,CENTRO MASSAGGI RILASSANTI NON ESTETICI,0.0,34.0,False,False,False,False,False,VIA,WATT GIACOMO,5,6
25,VLE BLIGNY N. 23 (Z.D. 5),VLE,BLIGNY,23,4045.0,5,SERVIZI DI CENTRI PER IL BENESSERE FISICO,0.0,149.0,False,False,False,False,False,VLE,BLIGNY,23,5
28,VLE SABOTINO N. 8 (Z.D. 5),VLE,SABOTINO,8,4046.0,5,ESTETICA A,0.0,46.0,False,False,False,False,False,VLE,SABOTINO,8,5


# *Outlier Detection*

In [167]:
# Compute Z-score on certain columns and trop outliers

# *Duplicate Detection*

In [168]:
# Drop exact duplicates

In [169]:
# Use 'Sorted Neighbourhood' to find possible duplicates

In [170]:
# Drop new found possible duplicates